# Clase 207 — Behavioral tests: invariance + directional + MFT + slice

Implementamos los 4 tipos sobre un modelo de income prediction (Adult dataset proxy con California Housing modificado para el ejemplo).

In [ ]:
import numpy as np, pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

data = fetch_california_housing(as_frame=True)
df = data.data.copy()
# Convertimos a clasificación binaria: 'high income block' = target > mediana
df['high'] = (data.target > data.target.median()).astype(int)
# Sintetizamos 'gender' aleatorio (sin info real → modelo NO debería usarlo)
rng = np.random.default_rng(42)
df['gender'] = rng.choice([0, 1], size=len(df))

FEATURES = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'gender']
Xtr, Xte, ytr, yte = train_test_split(df[FEATURES], df['high'], test_size=0.3, random_state=42)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xtr, ytr)
print('test acc:', model.score(Xte, yte))

## 1. Minimum Functionality Test (MFT)

In [ ]:
mft_cases = [
    # ('descripción', features, expected_label)
    ('Bloque opulento: ingreso muy alto, casas grandes', [15.0, 20, 8, 2, 1500, 3, 37.7, -122.4, 0], 1),
    ('Bloque humilde: ingreso bajo, casas chicas', [1.5, 30, 3, 1, 2000, 4, 34.0, -118.2, 0], 0),
    ('Suburbio próspero', [10.0, 15, 6, 2, 800, 2.5, 37.5, -122.0, 1], 1),
    ('Zona empobrecida densa', [2.0, 50, 3, 1.2, 3500, 5, 33.9, -118.3, 0], 0),
]
ok = sum(int(model.predict([feats])[0] == exp) for _, feats, exp in mft_cases)
print(f'MFT: {ok}/{len(mft_cases)} casos canónicos correctos')
assert ok == len(mft_cases), 'MFT falló — modelo no acierta casos triviales'
print('✅ MFT pasa.')

## 2. Invariance test — swap de gender

Gender es ruido random; el modelo no debería usarlo. Si `gender swap` cambia predicciones, hay un problema.

In [ ]:
sample = Xte.sample(500, random_state=0).copy()
pred_orig = model.predict(sample)
sample_swapped = sample.copy()
sample_swapped['gender'] = 1 - sample_swapped['gender']
pred_swapped = model.predict(sample_swapped)

agreement = (pred_orig == pred_swapped).mean()
print(f'invariance gender swap: {agreement:.4f}')
assert agreement >= 0.99, f'INV failed: gender swap cambia {(1 - agreement):.2%} de predicciones'
print('✅ INV pasa: gender no influye en la decisión.')

## 3. Directional test — subir `MedInc` no debería bajar P(high)

Monotonía esperada por dominio.

In [ ]:
sample = Xte.sample(500, random_state=0).copy()
p_orig = model.predict_proba(sample)[:, 1]
sample_up = sample.copy()
sample_up['MedInc'] = sample_up['MedInc'] * 1.5
p_up = model.predict_proba(sample_up)[:, 1]

violations = (p_up < p_orig - 0.05).sum()   # tolerancia de 5pp
print(f'directional: {len(sample) - violations}/{len(sample)} casos siguen la dirección esperada')
print(f'violaciones: {violations} ({violations / len(sample):.2%})')
assert violations < 0.05 * len(sample), 'DIR failed: subir MedInc baja P(high) en >5% de casos'
print('✅ DIR pasa.')

## 4. Slice-based testing

In [ ]:
Xte_with = Xte.copy()
Xte_with['high'] = yte.values
Xte_with['pred'] = model.predict(Xte)
Xte_with['age_bucket'] = pd.cut(Xte_with['HouseAge'], bins=[0, 15, 30, 50, 100], labels=['0-15', '15-30', '30-50', '50+'])
Xte_with['inc_bucket'] = pd.cut(Xte_with['MedInc'], bins=[0, 2, 4, 6, 20], labels=['low', 'mid-low', 'mid', 'high'])

slices = (Xte_with.groupby(['age_bucket', 'inc_bucket'], observed=True)
          .apply(lambda d: pd.Series({'acc': (d.pred == d.high).mean(), 'n': len(d)}))
          .reset_index())
slices = slices[slices['n'] >= 50]
worst = slices.nsmallest(5, 'acc')
print('worst slices (n>=50):')
print(worst.to_string(index=False))
print(f'\noverall acc: {(Xte_with.pred == Xte_with.high).mean():.4f}')
print(f'worst slice acc: {worst.iloc[0]["acc"]:.4f}')
# Gate: worst slice no debería estar a más de 20pp del overall
assert (Xte_with.pred == Xte_with.high).mean() - worst.iloc[0]['acc'] < 0.2, 'slice disparity demasiado alta'

## 5. Test suite empaquetado (`tests/test_model_behavior.py`)

In [ ]:
test_file = '''\
# tests/test_model_behavior.py
import joblib, pytest, pandas as pd, numpy as np

@pytest.fixture(scope="session")
def model():
    return joblib.load("model.pkl")

@pytest.fixture(scope="session")
def sample_test():
    return pd.read_parquet("data/test.parquet")

def test_mft_canonical_cases(model):
    cases = [...]   # cargá de un YAML versionado
    for desc, features, expected in cases:
        pred = int(model.predict([features])[0])
        assert pred == expected, f"MFT failed: {desc}"

def test_inv_gender_swap(model, sample_test):
    s = sample_test.sample(500, random_state=0).copy()
    p0 = model.predict(s)
    s["gender"] = 1 - s["gender"]
    p1 = model.predict(s)
    assert (p0 == p1).mean() >= 0.99

def test_dir_medinc_up(model, sample_test):
    s = sample_test.sample(500, random_state=0).copy()
    p0 = model.predict_proba(s)[:, 1]
    s["MedInc"] *= 1.5
    p1 = model.predict_proba(s)[:, 1]
    violations = (p1 < p0 - 0.05).mean()
    assert violations < 0.05

def test_slice_worst_within_20pp(model, sample_test):
    df = sample_test.copy()
    df["pred"] = model.predict(df.drop(columns=["label"]))
    overall = (df.pred == df.label).mean()
    df["slice"] = df["gender"].astype(str) + "_" + pd.cut(df["MedInc"], 4).astype(str)
    by_slice = df.groupby("slice").apply(lambda d: (d.pred == d.label).mean() if len(d) >= 50 else np.nan).dropna()
    assert (overall - by_slice.min()) < 0.20, f"worst slice {by_slice.min():.3f} vs overall {overall:.3f}"
'''
print(test_file)

## Ejercicio guiado

1. Hand-craft 20 MFT cases con un domain expert. Pónelos en `tests/mft_cases.yaml`. Tests los cargan.
2. Agregá un test de **adversarial robustness**: para 100 instancias, suma `ε * sign(gradient)` (FGSM). El modelo debería mantener accuracy >80%. Para sklearn: aproximá gradient con perturbaciones random.
3. Integrá los tests en GH Actions (Clase 197) como `required check`. Hacé un PR que rompa el INV gender (intencionalmente entrená un modelo con `gender` como feature útil) y confirmá que el CI bloquea el merge.
4. Corré `deepchecks` full suite. Identificá ≥1 problema que tu suite custom no detectó.
5. Para producción real: agregá un test que toma un sample del tráfico de la última semana y verifica que ≥95% de las predicciones siguen las direccionales esperadas.

## Conclusiones

- Accuracy alta + bugs sistemáticos en slices = modelo "bueno" que rompe en producción.
- INV / DIR / MFT son las 3 categorías que cubren ~90% de los bugs reales.
- Slice-based testing revela disparidades que el promedio esconde.
- Tests behavioral en CI = gate real, no opinión.
- **Fin de Parte 4**: data tests (206) + model tests (207) + monitoring (202) + shadow/canary/rollback (204) = 6 capas de protección.